# Final Project Analysis

This notebook brings together the Week 1 to Week 4 work on the Telco Customer Churn dataset. It reviews data quality, describes customer churn patterns, reports statistical tests, evaluates a logistic regression model, and outlines evidence-based retention recommendations.

The supplied Week 4 outputs are used as the model reference: test ROC-AUC 0.8417, recall 0.8209, precision 0.5049. Results are associations and predictions from this sample; they do not establish cause and effect.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

project_candidates = [Path.cwd(), Path.cwd() / "Week 5", Path.cwd() / "outputs" / "Week 5"]
PROJECT_DIR = next((p for p in project_candidates if (p / "Telco-Customer-Churn.csv").exists()), Path.cwd())
FIGURE_DIR = PROJECT_DIR / "week5_figures"
FIGURE_DIR.mkdir(exist_ok=True)


In [ ]:
# Load and clean the Telco Customer Churn dataset
data_path = PROJECT_DIR / "Telco-Customer-Churn.csv"
if not data_path.exists():
    raise FileNotFoundError("Place Telco-Customer-Churn.csv beside this notebook.")

df = pd.read_csv(data_path)
df.columns = df.columns.str.strip()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
zero_tenure = df["tenure"].eq(0)
df.loc[zero_tenure & df["TotalCharges"].isna(), "TotalCharges"] = 0
df = df.drop_duplicates().copy()

print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing values after cleaning:", int(df.isna().sum().sum()))
df.head()


## Customer Churn Overview

Churn is the binary outcome for this project. The overall churn percentage gives context for evaluating the later model, because stayed customers are the larger class.


In [ ]:
churn_counts = df["Churn"].value_counts().reindex(["No", "Yes"])
churn_percent = df["Churn"].value_counts(normalize=True).reindex(["No", "Yes"]) * 100
print(churn_counts)
print("\nChurn percentages:")
print(churn_percent.round(2))

ax = churn_counts.plot(kind="bar", color=["#4c9f70", "#d97757"], figsize=(7, 4.5))
ax.set(title="Customer Churn Counts", xlabel="Churn outcome", ylabel="Number of customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "week5_churn_counts.png", dpi=200, bbox_inches="tight")
plt.show()


## Churn by Contract Type

Week 2 and Week 3 identified contract type as an important descriptive pattern. The following chart compares observed churn rates. These percentages describe the dataset and should not be read as a causal effect of contract choice.


In [ ]:
contract_rate = (
    df.assign(ChurnFlag=df["Churn"].eq("Yes"))
      .groupby("Contract", observed=True)["ChurnFlag"]
      .mean()
      .reindex(["Month-to-month", "One year", "Two year"])
)
print((contract_rate * 100).round(2).rename("Churn rate (%)"))

ax = (contract_rate * 100).plot(kind="bar", color="#2878a5", figsize=(7, 4.5))
ax.set(title="Observed Churn Rate by Contract Type", xlabel="Contract type", ylabel="Customers churned (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "week5_contract_churn_rates.png", dpi=200, bbox_inches="tight")
plt.show()


## Monthly Charges and Tenure Patterns

The Week 2 visual analysis compared tenure and monthly charges by churn outcome. Group averages and the box plot below make these differences visible while leaving the causal question open.


In [ ]:
charge_summary = df.groupby("Churn")["MonthlyCharges"].agg(["count", "mean", "median", "std"])
tenure_summary = df.groupby("Churn")["tenure"].agg(["count", "mean", "median", "std"])
print("Monthly charges by churn:")
print(charge_summary.round(2))
print("\nTenure by churn:")
print(tenure_summary.round(2))

groups = [df.loc[df["Churn"].eq("Yes"), "MonthlyCharges"],
          df.loc[df["Churn"].eq("No"), "MonthlyCharges"]]
plt.figure(figsize=(7, 4.5))
plt.boxplot(groups, showfliers=False)
plt.xticks([1, 2], ["Churned", "Stayed"])
plt.title("Monthly Charges by Churn Outcome")
plt.ylabel("Monthly charges")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "week5_monthly_charges_by_churn.png", dpi=200, bbox_inches="tight")
plt.show()


## Week 3 Statistical Tests

The chi-square test checks whether contract type and churn are independent. Welch's t-test compares the mean monthly charges of the churned and stayed groups without assuming equal variances.


In [ ]:
contract_table = pd.crosstab(df["Contract"], df["Churn"])
chi2, chi_p, chi_dof, expected = stats.chi2_contingency(contract_table, correction=False)
n = contract_table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * min(contract_table.shape[0] - 1, contract_table.shape[1] - 1)))
print(f"Chi-square({chi_dof}) = {chi2:.4f}; p = {chi_p:.4g}; Cramer's V = {cramers_v:.4f}")

churned = df.loc[df["Churn"].eq("Yes"), "MonthlyCharges"].astype(float)
stayed = df.loc[df["Churn"].eq("No"), "MonthlyCharges"].astype(float)
welch = stats.ttest_ind(churned, stayed, equal_var=False)
print(f"Churned mean = ${churned.mean():.2f}; stayed mean = ${stayed.mean():.2f}")
print(f"Welch t = {welch.statistic:.4f}; p = {welch.pvalue:.4g}")


The Week 3 report gave chi-square(2) = 1,184.60 with p = 5.86 × 10⁻²⁵⁸ and Cramér's V = 0.410. Its Welch test gave a mean-charge difference of $13.18 (95% CI: $11.77 to $14.58), p = 8.59 × 10⁻⁷³, and Cohen's d = 0.446. The tests support group differences in this sample; they do not prove that contract type or price causes churn.

## Week 4 Logistic Regression Model

This section reproduces the class-weighted logistic regression workflow. Preprocessing is part of the pipeline so that imputers, scaling, and category encoding are fitted on training data only.


In [ ]:
y = df["Churn"].astype(str).str.strip().map({"No": 0, "Yes": 1}).astype(int)
X = df.drop(columns=["Churn", "customerID"], errors="ignore")

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

def stratified_split(X, y, test_size=0.20, random_state=42):
    rng = np.random.default_rng(random_state)
    train_positions, test_positions = [], []
    for label in sorted(y.unique()):
        positions = np.flatnonzero(y.to_numpy() == label)
        rng.shuffle(positions)
        n_test = int(np.ceil(test_size * len(positions)))
        test_positions.extend(positions[:n_test])
        train_positions.extend(positions[n_test:])
    train_positions, test_positions = np.asarray(train_positions), np.asarray(test_positions)
    rng.shuffle(train_positions)
    rng.shuffle(test_positions)
    return (X.iloc[train_positions], X.iloc[test_positions],
            y.iloc[train_positions], y.iloc[test_positions])

X_train, X_test, y_train, y_test = stratified_split(X, y)
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced",
                                       solver="lbfgs", random_state=42)),
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]


In [ ]:
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_score),
}
print("Rows:", len(X), "| training:", len(X_train), "| test:", len(X_test))
print(pd.Series(metrics).round(4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Stayed", "Churned"], zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["Stayed", "Churned"]).plot(
    cmap="Blues", colorbar=False, values_format="d"
)
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "week5_model_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_score)
plt.figure(figsize=(6.5, 5))
plt.plot(fpr, tpr, label=f"Logistic regression (AUC={metrics['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "--", label="Chance")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC Curve on the Held-Out Test Set")
plt.legend(loc="lower right")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "week5_model_roc_curve.png", dpi=200, bbox_inches="tight")
plt.show()


## Strategic Recommendations

1. **Review month-to-month accounts first.** Their observed churn rate is 42.71%, compared with 11.27% for one-year and 2.83% for two-year contracts. Offer clear plan-fit reviews and optional retention choices; do not treat contract length as a cause.
2. **Improve early customer support.** Churned customers have lower average tenure. Test onboarding and early-service follow-up with a control group.
3. **Investigate billing and value perception.** Churned customers pay higher average monthly charges, and electronic-check customers show 45.29% observed churn. Review billing experience and plan clarity before offering broad discounts.
4. **Use the model to prioritize human review.** Week 4 recall was 82.09%, while precision was 50.49%. The false-positive volume should be considered against outreach capacity and cost.
5. **Measure impact with a controlled pilot.** Compare 30-day and 90-day retention, incremental retained customers, offer costs, contribution margin, complaints, and results by customer segment.

## Key Findings and Limitations

- The dataset contains 7,043 records and 21 columns; 1,869 customers (26.54%) are marked as churned.
- Contract type is associated with churn, and monthly charges differ between churn groups in the statistical tests.
- The logistic regression model has useful ranking performance on one held-out split (ROC-AUC 0.8417), but the model is not evidence of causation.
- The data represent one observational sample. The model needs cross-validation, time-based validation, threshold selection, calibration, fairness review, and monitoring before business use.
- No intervention cost or experimental result was supplied, so this project does not claim a dollar impact.
